In [ ]:
!tar -xzf /content/mnist_png.tar.gz

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

import torch.nn as nn

from torch.utils.data import Dataset,DataLoader,random_split
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm
import pathlib


In [ ]:
test_path = pathlib.Path('/content/mnist_png/testing')
train_path = pathlib.Path('/content/mnist_png/training')

In [ ]:
transform = T.Compose([
    T.Resize(32),
    T.ToTensor()
])

In [ ]:
class MNISTDataset(Dataset):

    def __init__(self, path, transform = None):
        # int for cross entryp loss
        self.path_list = list(path.glob('*/*.png'))
        self.transform = transform

    def __getitem__(self,index):
        img = Image.open(self.path_list[index])
        label = (self.path_list[index]).parts[-2]

        if transform != None:
            img = transform(img)

        return img, int(label)

    def __len__(self):
        return len(self.path_list)

In [ ]:
train_dataset = MNISTDataset(train_path,transform)
test_dataset = MNISTDataset(test_path,transform)

In [ ]:
train_data = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_data = DataLoader(test_dataset,batch_size=64,shuffle=True)

In [ ]:
class LeNet(nn.Module):
  def __init__(self):
    super(LeNet, self).__init__()
    # 32X32X1
    self.conv1 = nn.Conv2d(in_channels = 1, out_channels = 6,
                           kernel_size = 5, stride = 1, padding = 0)
    # 28X28X6
    self.conv2 = nn.Conv2d(in_channels = 6, out_channels = 16,
                           kernel_size = 5, stride = 1, padding = 0)
    # 14X14X6
    self.conv3 = nn.Conv2d(in_channels = 16, out_channels = 120,
                           kernel_size = 5, stride = 1, padding = 0)
    # 10X10X16

    # AVG POLL 5X5X16

    self.linear1 = nn.Linear(120, 84)
    self.linear2 = nn.Linear(84, 10)
    self.tanh = nn.Tanh()
    self.avgpool = nn.AvgPool2d(kernel_size = 2, stride = 2)

  def forward(self, x):
    # 32X32X1
    x = self.conv1(x)
    x = self.tanh(x)
    # 28X28X6
    x = self.avgpool(x)
    # 14x14x6
    x = self.conv2(x)
    # 10x10x16
    x = self.tanh(x)
    x = self.avgpool(x)
    # 5x5x16
    x = self.conv3(x)
    # 1x1x120

    x = self.tanh(x)

    x = x.reshape(x.shape[0], -1)
    # 120
    x = self.linear1(x)
    # 84
    x = self.tanh(x)
    # 84
    x = self.linear2(x)
    # 10
    return x

In [ ]:
model = LeNet()

In [ ]:
# loss and optimizer
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
print(torch.cuda.is_available())

False


In [ ]:
class Trainer:
    def __init__(self, model, train_dataloader, test_dataloader, criterion, optimizer, epochs=3):
        self.model = model
        self.train_dataloader = train_dataloader
        self.test_dataloader = test_dataloader
        self.criterion = criterion
        self.optimizer = optimizer

        self.epochs = epochs
        self.history = {'loss':[],'acc':[],'val_loss':[],'val_acc':[]}

    def train_loop(self):
        sum_loss = 0
        sum_accuracy = 0
        n = len(self.train_dataloader)
        for i, (data,label) in enumerate(tqdm(self.train_dataloader)):
            data = data.to(device)
            label = label.to(device)
            # prediction model
            output = self.model(data)
            # find loss
            loss = self.criterion(output, label)

            sum_loss += loss.item()
            n_corrects = (output.argmax(axis=1)==label).sum().item()
            sum_accuracy += n_corrects/label.size(0)

            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()


        train_loss = sum_loss/n
        train_accuracy = sum_accuracy/n

        self.history['loss'].append(train_loss)
        self.history['acc'].append(train_accuracy)

        return train_loss, train_accuracy

    def validation_loop(self):
        sum_loss = 0
        sum_accuracy = 0
        n = len(self.test_dataloader)
        for i, (data,label) in enumerate(tqdm(self.test_dataloader)):
            data = data.to(device)
            label = label.to(device)
            # prediction model
            output = self.model(data)
            # find loss
            loss = self.criterion(output, label)
            n_corrects = (output.argmax(axis=1)==label).sum().item()

            sum_loss += loss.item()
            sum_accuracy += n_corrects/label.size(0)

        val_loss = sum_loss/n
        val_accuracy = sum_accuracy/n

        self.history['val_loss'].append(val_loss)
        self.history['val_acc'].append(val_accuracy)

        return val_loss, val_accuracy

    def train(self):
        for epoch in range(self.epochs):
            train_loss, train_acc = self.train_loop()
            val_loss, val_acc = self.validation_loop()
            print()
            print(f'Epoch[{epoch+1}/{self.epochs}] \t train_loss: {train_loss:.5f}, train_acc: {train_acc:.2f} \t val_loss: {val_loss:.5f} \t val_acc: {val_acc:.2}')


In [ ]:
trainer = Trainer(
    model = model.to(device),
    train_dataloader=train_data,
    test_dataloader=test_data,
    criterion=criterion,
    optimizer=optimizer,
    epochs=5,
)

In [ ]:
trainer.train()

100%|██████████| 157/157 [00:06<00:00, 25.32it/s]



Epoch[1/3] 	 train_loss: 0.04663, train_acc: 0.99 	 val_loss: 0.04749 	 val_acc: 0.98


100%|██████████| 157/157 [00:05<00:00, 28.89it/s]



Epoch[2/3] 	 train_loss: 0.03665, train_acc: 0.99 	 val_loss: 0.05119 	 val_acc: 0.99


100%|██████████| 157/157 [00:05<00:00, 28.32it/s]


Epoch[3/3] 	 train_loss: 0.02955, train_acc: 0.99 	 val_loss: 0.04589 	 val_acc: 0.99


In [ ]:
print(trainer.history['loss'])

[0.04662702579139623, 0.03664683887946245, 0.029548358262499432]
